In [16]:
import json, os, pickle
import numpy as np
from tqdm.notebook import tqdm
import pandas as pd

import torch
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
import seaborn as sns
import qiskit.circuit.random
import torch, random
from torch.utils.data import Dataset, DataLoader, TensorDataset
from torch.optim.lr_scheduler import ReduceLROnPlateau
import torch.nn as nn
from qiskit.qpy import load

import numpy as np
import json, os, pickle
from tqdm import tqdm
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
from qiskit.qasm2 import dump

from io import StringIO

from qiskit import QuantumCircuit

import sys
sys.path.append('../tutorials/')
from mlp import encode_data, encode_data_v2_ecr

In [55]:
def load_circuits(data_dir, json_file_dir):

    circuits = []
    noisy_expected_values = []
    noiseless_expected_values = []

    with open(json_file_dir) as json_file:
        json_data = json.load(json_file)

    circuit_names = list(json_data.keys())
    # print(circuit_names)

    for circuit_name in tqdm(circuit_names, leave=True):

        file_path = os.path.join(data_dir, circuit_name)
        try:
            with open(file_path, "rb") as f:
                circuit = load(f)  # returns a list of QuantumCircuit objects
                if len(json_data[circuit_name]["z_noisy"][:5]) != 5:
                    continue
                else:
                    circuits.append(circuit[0])
                    noisy_expected_values.append(json_data[circuit_name]["z_noisy"][:5])
                    noiseless_expected_values.append(json_data[circuit_name]["z_ideal"][:5])
        except Exception as e:
            print(f"⚠️ Error loading {file}: {e}")
            continue

    return circuits, noisy_expected_values, noiseless_expected_values


def load_circuits_real_data(data_dir, real_json_file_dir, ideal_json_path):

    circuits = []
    noisy_expected_values = []
    noiseless_expected_values = []

    all_circuits = os.listdir(data_dir)
    # print(all_circuits)

    with open(ideal_json_path) as json_file:
        ideal_json_data = json.load(json_file)

    # circuit_names = list(json_data.keys())
    # print(circuit_names)

    for circuit_name in tqdm(all_circuits, leave=True):

        real_file_path = os.path.join(real_json_file_dir,f"{circuit_name}_hardware_shots.json")
        # sim_file_path = os.path.join(real_json_file_dir,f"{circuit_name}__ideal_shots.json")

        try:
            with open(real_file_path) as real_json_file:
                real_data_loaded = json.load(real_json_file)
            # with open(sim_file_path) as ideal_json_file:
            #     ideal_data_loaded = json.load(ideal_json_file)
        
            file_path = os.path.join(data_dir, circuit_name)
        

            with open(file_path, "rb") as f:
                circuit = load(f)  # returns a list of QuantumCircuit objects
                if len(real_data_loaded["expected_z"][:5]) != 5:
                    continue
                else:
                    circuits.append(circuit[0])
                    noisy_expected_values.append(real_data_loaded["expected_z"][:5])
                    noiseless_expected_values.append(ideal_json_data[circuit_name]["z_ideal"][:5])
        except Exception as e:
            print(f"⚠️ Error loading: {e}")
            continue

    return circuits, noisy_expected_values, noiseless_expected_values


In [56]:
data_dir = "../../../andrew/ExecutionResults/StoredCircuits/"
json_file_dir = "../../andrew_experiments/extracted_data/"
ideal_json_apth = "./z_expectations.json"

circuits, noisy_expected_values, noiseless_expected_values = load_circuits_real_data(data_dir, json_file_dir, ideal_json_apth)

  2%|█▋                                                                                                     | 112/7004 [00:00<00:06, 1081.98it/s]

⚠️ Error loading: [Errno 2] No such file or directory: '../../andrew_experiments/extracted_data/ea11a710-d03d-4d65-be29-0c7a5949fee2.qpy_hardware_shots.json'
⚠️ Error loading: [Errno 2] No such file or directory: '../../andrew_experiments/extracted_data/8d0af168-21d5-4b59-8495-183145b19d61.qpy_hardware_shots.json'
⚠️ Error loading: [Errno 2] No such file or directory: '../../andrew_experiments/extracted_data/a31d466e-951d-422d-ba34-90ea0e6512bb.qpy_hardware_shots.json'
⚠️ Error loading: [Errno 2] No such file or directory: '../../andrew_experiments/extracted_data/ff5140d6-c452-4630-8bf2-8d26cc1a2fbe.qpy_hardware_shots.json'
⚠️ Error loading: [Errno 2] No such file or directory: '../../andrew_experiments/extracted_data/81917f2f-c962-48f9-8a11-c8afea0053d1.qpy_hardware_shots.json'
⚠️ Error loading: [Errno 2] No such file or directory: '../../andrew_experiments/extracted_data/0c1b3603-4e2a-47a1-b923-78eb694970d5.qpy_hardware_shots.json'
⚠️ Error loading: [Errno 2] No such file or director

  6%|██████▍                                                                                                | 438/7004 [00:00<00:04, 1446.16it/s]

⚠️ Error loading: [Errno 2] No such file or directory: '../../andrew_experiments/extracted_data/220c28ab-6235-4f75-ade4-66bceec52cdf.qpy_hardware_shots.json'
⚠️ Error loading: [Errno 2] No such file or directory: '../../andrew_experiments/extracted_data/6d47bc58-1e75-4779-820f-534f5fc95129.qpy_hardware_shots.json'
⚠️ Error loading: [Errno 2] No such file or directory: '../../andrew_experiments/extracted_data/2d46d29b-b83c-41ae-b0a8-035bde81e741.qpy_hardware_shots.json'
⚠️ Error loading: [Errno 2] No such file or directory: '../../andrew_experiments/extracted_data/bd6606d3-95a7-47a2-a173-1b33c8640a37.qpy_hardware_shots.json'
⚠️ Error loading: [Errno 2] No such file or directory: '../../andrew_experiments/extracted_data/3c24354e-2588-4a4d-a4a6-730a85035e83.qpy_hardware_shots.json'
⚠️ Error loading: [Errno 2] No such file or directory: '../../andrew_experiments/extracted_data/fce4b629-fc76-468f-a84a-1be0f00d0161.qpy_hardware_shots.json'
⚠️ Error loading: [Errno 2] No such file or director

 11%|███████████                                                                                            | 750/7004 [00:00<00:04, 1472.73it/s]

...

⚠️ Error loading: [Errno 2] No such file or directory: '../../andrew_experiments/extracted_data/846cbfd7-e520-4f33-9197-1d024c66b036.qpy_hardware_shots.json'
⚠️ Error loading: [Errno 2] No such file or directory: '../../andrew_experiments/extracted_data/f8641b14-65dd-44b6-abd6-1f193f4090e8.qpy_hardware_shots.json'
⚠️ Error loading: [Errno 2] No such file or directory: '../../andrew_experiments/extracted_data/33b2ce39-0edf-4daa-9627-3306fd36008f.qpy_hardware_shots.json'
⚠️ Error loading: [Errno 2] No such file or directory: '../../andrew_experiments/extracted_data/fda714a6-a234-49ec-b133-12c9c5058760.qpy_hardware_shots.json'
⚠️ Error loading: [Errno 2] No such file or directory: '../../andrew_experiments/extracted_data/df0cdf82-a890-4a5c-8b11-c2484f43b8e7.qpy_hardware_shots.json'
⚠️ Error loading: [Errno 2] No such file or directory: '../../andrew_experiments/extracted_data/3ffd54dc-3956-4f8f-ba5a-7cd4da54347a.qpy_hardware_shots.json'
⚠️ Error loading: [Errno 2] No such file or director

 98%|████████████████████████████████████████████████████████████████████████████████████████████████████▋  | 6845/7004 [00:09<00:00, 528.56it/s]

⚠️ Error loading: [Errno 2] No such file or directory: '../../andrew_experiments/extracted_data/dfc7f7af-0b88-4936-b36c-373f17f499ec.qpy_hardware_shots.json'
⚠️ Error loading: [Errno 2] No such file or directory: '../../andrew_experiments/extracted_data/41717a18-fc3b-400b-92cf-8f1fc78dce38.qpy_hardware_shots.json'
⚠️ Error loading: [Errno 2] No such file or directory: '../../andrew_experiments/extracted_data/b84a3318-93f1-416c-8717-cf7635f79453.qpy_hardware_shots.json'
⚠️ Error loading: [Errno 2] No such file or directory: '../../andrew_experiments/extracted_data/bf769ba2-d76d-4294-ad2f-abbb7360bc67.qpy_hardware_shots.json'
⚠️ Error loading: [Errno 2] No such file or directory: '../../andrew_experiments/extracted_data/c701a04d-6e36-4b7d-8c33-028ea35f000d.qpy_hardware_shots.json'
⚠️ Error loading: [Errno 2] No such file or directory: '../../andrew_experiments/extracted_data/3e19bac2-8c47-4906-99c1-3b4835d78f14.qpy_hardware_shots.json'
⚠️ Error loading: [Errno 2] No such file or director

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 7004/7004 [00:09<00:00, 754.78it/s]

⚠️ Error loading: [Errno 2] No such file or directory: '../../andrew_experiments/extracted_data/8b32cd9b-c51f-4846-9e77-26b136d90694.qpy_hardware_shots.json'
⚠️ Error loading: [Errno 2] No such file or directory: '../../andrew_experiments/extracted_data/238f07f0-6fe6-4756-8505-e12b3c7471b7.qpy_hardware_shots.json'
⚠️ Error loading: [Errno 2] No such file or directory: '../../andrew_experiments/extracted_data/f019eade-0b0d-4ab4-afcc-adce7f46365d.qpy_hardware_shots.json'
⚠️ Error loading: [Errno 2] No such file or directory: '../../andrew_experiments/extracted_data/bee578e7-b64a-4d2d-a4ce-d76cd5d8732e.qpy_hardware_shots.json'
⚠️ Error loading: [Errno 2] No such file or directory: '../../andrew_experiments/extracted_data/6f4fc944-552e-4ea9-b56c-d487942f8eaf.qpy_hardware_shots.json'
⚠️ Error loading: [Errno 2] No such file or directory: '../../andrew_experiments/extracted_data/c3ef7d67-f1de-4c9a-b982-c23cc493421c.qpy_hardware_shots.json'
⚠️ Error loading: [Errno 2] No such file or director

In [54]:
len(circuits)

4852

In [41]:
all_circuits = os.listdir(data_dir)

circuit = all_circuits[0]
print(circuit)

json_path = os.path.join(json_file_dir, f"{circuit}_hardware_shots.json")

with open(json_path) as json_file:
    data = json.load(json_file)

print(data)

9442e55b-2c38-4287-85c0-61af0dcc7dbc.qpy
{'counts': {'0111010110': 11, '1000011001': 2, '0101000000': 17, '0011001010': 13, '1011111111': 6, '1010011010': 8, '0010101101': 2, '0110100000': 8, '1011010011': 13, '1101010001': 21, '0111101101': 2, '1101010011': 25, '0111101000': 8, '1010010111': 10, '0001001100': 9, '1101001110': 13, '0101110101': 7, '1101000000': 24, '0101010101': 12, '1000000101': 9, '0010111010': 8, '1101001101': 7, '0101010100': 7, '1100110011': 6, '1100100110': 4, '0111010010': 19, '1100101001': 3, '1101000100': 20, '1000110110': 11, '1101101110': 8, '0000110001': 4, '0010011000': 14, '0100000101': 12, '0110110011': 9, '0100110101': 10, '0100010111': 14, '1100100011': 9, '0110111101': 3, '1110010000': 17, '0101001001': 7, '0101011100': 9, '0000101001': 6, '0010111111': 7, '0010111001': 7, '1100101011': 8, '1100100101': 4, '0010010000': 16, '1111100000': 9, '1000011110': 6, '0111000010': 13, '1101110110': 10, '0101101010': 12, '0010110011': 4, '1010001111': 7, '101001

In [57]:
def count_gates_by_rotation_angle(circuit, bin_size):
    angles = []
    for instr, qargs, cargs in circuit.data:
        if instr.name in ['rx', 'ry', 'rz'] and len(qargs) == 1:
            angles += [float(instr.params[0])]
    bin_edges = np.arange(-2 * np.pi, 2 * np.pi + bin_size, bin_size)
    counts, _ = np.histogram(angles, bins=bin_edges)
    bin_labels = [f"{left:.2f} to {right:.2f}" for left, right in zip(bin_edges[:-1], bin_edges[1:])]
    angle_bins = {label: count for label, count in zip(bin_labels, counts)}
    return list(angle_bins.values())


def recursive_dict_loop(my_dict, parent_key=None, out=None, target_key1=None, target_key2=None):
    if out is None: out = []

    for key, val in my_dict.items():
        if isinstance(val, dict):
            recursive_dict_loop(val, key, out, target_key1, target_key2)
        else:
            if parent_key and target_key1 in str(parent_key) and key == target_key2:
                out += [val]
    return out or 0.


def encode_data_v2_ecr_CLIP_encoder(circuits, 
                                    ideal_exp_vals, 
                                    noisy_exp_vals, 
                                    obs_size, 
                                    tokenizer,
                                    text_model,
                                    device,
                                    max_len = 77,
                                    meas_bases=None, 
                                    two_q_gate='ecr'):
    
    if isinstance(noisy_exp_vals[0], list) and len(noisy_exp_vals[0]) == 1:
        noisy_exp_vals = [x[0] for x in noisy_exp_vals]

    if meas_bases is None:
        meas_bases = [[]]

    gates_set = [two_q_gate] + ['sx', 'x', 'id', 'rz']

    vec = []

    bin_size = 0.025 * np.pi
    num_angle_bins = int(np.ceil(4 * np.pi / bin_size))

    X = torch.zeros([len(circuits), 512 + len(vec) + len(gates_set) + num_angle_bins + obs_size + len(meas_bases[0])]) #512 for CLIP Embedding

    embedding_slice = slice(0,512)
    vec_slice = slice(512, 512+len(vec))
    gate_counts_slice = slice(512+len(vec), 512+len(vec)+len(gates_set))
    angle_bins_slice = slice(512+len(vec)+len(gates_set), 512+len(vec)+len(gates_set)+num_angle_bins)
    exp_val_slice = slice(512+len(vec)+len(gates_set)+num_angle_bins, 512+len(vec)+len(gates_set)+num_angle_bins+obs_size)
    meas_basis_slice = slice(512+len(vec)+len(gates_set)+num_angle_bins+obs_size, len(X[0]))

    # X[:, vec_slice] = vec[None, :]

    for i, circ in enumerate(tqdm(circuits)):
        qasm_buffer = StringIO()
        dump(circ, qasm_buffer)
        qasm_code = qasm_buffer.getvalue()
        
        circuit_qasm = qasm_code

        tokens = tokenizer(circuit_qasm, return_tensors="pt", truncation=False, padding=False)
        input_ids = tokens["input_ids"][0]  # remove batch dimension
        
        chunks = [input_ids[i:i + max_len] for i in range(0, len(input_ids), max_len)]
        
        # Encode each chunk and collect pooled outputs
        embeddings = []
        
        for chunk in chunks:
            chunk = chunk.unsqueeze(0).to(device)
            with torch.no_grad():
                output = text_model(input_ids=chunk)
                pooled = output.pooler_output  # shape: (1, hidden_dim)
            embeddings.append(pooled.cpu())  # keep CPU to save GPU memory
        
        # Combine embeddings (mean pooling)
        final_embedding = torch.mean(torch.stack(embeddings), dim=0)

        
        X[i, embedding_slice] = torch.tensor(final_embedding)

    
    for i, circ in enumerate(circuits):
        gate_counts_all = circ.count_ops()
        X[i, gate_counts_slice] = torch.tensor(
            [gate_counts_all.get(key, 0) for key in gates_set]
        ) * 0.01  # put it in the same order of magnitude as the expectation values

    for i, circ in enumerate(circuits):
        gate_counts = count_gates_by_rotation_angle(circ, bin_size)
        X[i, angle_bins_slice] = torch.tensor(gate_counts) * 0.01  # put it in the same order of magnitude as the expectation values

        if obs_size > 1: assert len(noisy_exp_vals[i]) == obs_size
        elif obs_size == 1: assert isinstance(noisy_exp_vals[i], float)

        X[i, exp_val_slice] = torch.tensor(noisy_exp_vals[i])

    if meas_bases != [[]]:
        assert len(meas_bases) == len(circuits)
        for i, basis in enumerate(meas_bases):
            X[i, meas_basis_slice] = torch.tensor(basis)

    y = torch.tensor(ideal_exp_vals, dtype=torch.float32)

    return X, y

In [58]:
num_circ_per_step = 50
k = train_test_split = 40
train_circuits = []
train_ideal_vals = []
train_noisy_vals = []
test_circuits = []
test_ideal_vals = []
test_noisy_vals = []
test_Js = []
for start_each_step in list(range(len(circuits))[::num_circ_per_step]):
    train_circuits += circuits[start_each_step:start_each_step+k]
    train_ideal_vals += noiseless_expected_values[start_each_step:start_each_step+k]
    train_noisy_vals += noisy_expected_values[start_each_step:start_each_step+k]
    test_circuits += circuits[start_each_step+k:start_each_step+num_circ_per_step]
    test_ideal_vals += noiseless_expected_values[start_each_step+k:start_each_step+num_circ_per_step]
    test_noisy_vals += noisy_expected_values[start_each_step+k:start_each_step+num_circ_per_step]
    # test_Js += Js[start_each_step+k:start_each_step+num_circ_per_step]

In [59]:
print(len(train_circuits), len(train_ideal_vals), len(train_noisy_vals))
print(len(test_circuits), len(test_ideal_vals), len(test_noisy_vals))

3882 3882 3882
970 970 970


In [60]:
normal_X_train, normal_y_train = encode_data_v2_ecr(train_circuits, train_ideal_vals, train_noisy_vals, obs_size=5)
normal_X_test, normal_y_test = encode_data_v2_ecr(test_circuits, test_ideal_vals, test_noisy_vals, obs_size=5)

In [61]:
print(normal_X_train.shape, normal_y_train.shape)
print(normal_X_test.shape, normal_y_test.shape)

torch.Size([3882, 170]) torch.Size([3882, 5])
torch.Size([970, 170]) torch.Size([970, 5])


In [62]:
from transformers import CLIPTokenizer, CLIPTextModel
import torch
from tqdm import tqdm
import time

In [63]:
# Setup device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load tokenizer and model
tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-base-patch32")
text_model = CLIPTextModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
text_model.eval()

Using device: cuda


CLIPTextModel(
  (text_model): CLIPTextTransformer(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 512)
      (position_embedding): Embedding(77, 512)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=512, out_features=2048, bias=True)
            (fc2): Linear(in_features=2048, out_features=512, bias=True)
          )
          (layer_norm2): LayerNorm((512,), eps=1e

In [64]:
X_train, y_train = encode_data_v2_ecr_CLIP_encoder(train_circuits, 
                                                   train_ideal_vals, 
                                                   train_noisy_vals, 
                                                   tokenizer=tokenizer,
                                                   text_model=text_model,
                                                   device=device,
                                                   obs_size=5)

  0%|                                                                                                                   | 0/3882 [00:00<?, ?it/s]Token indices sequence length is longer than the specified maximum sequence length for this model (1453 > 77). Running this sequence through the model will result in indexing errors
/tmp/ipykernel_1389793/4169997577.py:86: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X[i, embedding_slice] = torch.tensor(final_embedding)
100%|████████████████████████████████████████████████████████████████████████████████████████████████████████| 3882/3882 [03:08<00:00, 20.54it/s]
/tmp/ipykernel_1389793/4169997577.py:3: DeprecationWarning: Treating CircuitInstruction as an iterable is deprecated legacy behavior since Qiskit 1.2, and will be removed in Qiskit 3.0. Instead, use the `operation`, `qubits` and `clbi

In [65]:
torch.save(X_train, 'CLIP_Average_X_train_Andrew_real_qpu.pt')
torch.save(y_train, 'CLIP_Average_y_train_Andrew_real_qpu.pt')



In [66]:
X_test, y_test = encode_data_v2_ecr_CLIP_encoder(test_circuits, 
                                                 test_ideal_vals, 
                                                 test_noisy_vals, 
                                                 tokenizer=tokenizer,
                                                 text_model=text_model,
                                                 device=device,
                                                 obs_size=5)

torch.save(X_test, 'CLIP_Average_X_test_Andrew_real_qpu.pt')
torch.save(y_test, 'CLIP_Average_y_test_Andrew_real_qpu.pt')

  0%|                                                                                                                    | 0/970 [00:00<?, ?it/s]/tmp/ipykernel_1389793/4169997577.py:86: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X[i, embedding_slice] = torch.tensor(final_embedding)
100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████| 970/970 [00:47<00:00, 20.31it/s]
/tmp/ipykernel_1389793/4169997577.py:3: DeprecationWarning: Treating CircuitInstruction as an iterable is deprecated legacy behavior since Qiskit 1.2, and will be removed in Qiskit 3.0. Instead, use the `operation`, `qubits` and `clbits` named attributes.
  for instr, qargs, cargs in circuit.data:
